# W10: 공급업체 소통 자동화

이슈 유형별로 공문 초안을 Gemini로 생성해 텍스트 파일로 저장하고 ZIP으로 묶습니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

import zipfile

issues = pd.DataFrame({
    "업체명": ["A물류", "B식자재", "C포장"],
    "이슈유형": ["납품지연", "불량", "수량부족"],
    "사유": ["교통지연", "박스파손", "재고부족"],
    "요청사항": ["긴급 재발송", "교체요청", "우선발주"]
})

Path("vendor_letters").mkdir(exist_ok=True)
for r in issues.itertuples():
    prompt = f"업체:{r.업체명}, 유형:{r.이슈유형}, 사유:{r.사유}, 요청:{r.요청사항} 공문 3문장 생성"
    txt = use_gemini(prompt, f"[{r.업체명}] {r.이슈유형} 관련 조치를 요청드립니다. {r.요청사항} 건 빠른 회신 바랍니다.")
    Path("vendor_letters").joinpath(f"{r.업체명}_{r.이슈유형}.txt").write_text(txt, encoding="utf-8")

with zipfile.ZipFile("vendor_letter_pack.zip", "w") as z:
    for f in Path("vendor_letters").glob("*.txt"):
        z.write(f, arcname=f.name)

print("vendor_letter_pack.zip 생성")
